# Day 3 Lab – Document Q&A Assistant

**Objective:** Build a RAG-based question answering assistant that loads documents, chunks them, embeds and indexes them in a vector store, and answers questions using retrieved context and an LLM.

**Prerequisites:** Day 3 Modules 1 (LangChain) and 2 (RAG). OpenAI API key required.

**Duration:** ~45–60 minutes.

**Scenario:** Internal knowledge base. Employees ask questions in natural language; the assistant answers from company documents only — never from general knowledge.

## Pipeline diagram

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                     DOCUMENT Q&A ASSISTANT PIPELINE                          │
│                                                                             │
│  ┌────────────┐   ┌────────────┐   ┌────────────┐   ┌────────────┐         │
│  │ Block 1    │──▶│ Block 2    │──▶│ Block 3    │──▶│ Block 4    │         │
│  │ Load docs  │   │ Chunk docs │   │ Embed &    │   │ Retriever  │         │
│  │            │   │            │   │ index      │   │            │         │
│  └────────────┘   └────────────┘   └────────────┘   └─────┬──────┘         │
│                                                            │                │
│                                                            ▼                │
│                                    ┌────────────┐   ┌────────────┐         │
│                                    │ Block 6    │◀──│ Block 5    │         │
│                                    │ Display    │   │ Prompt +   │         │
│                                    │ answer +   │   │ LLM        │         │
│                                    │ sources    │   │            │         │
│                                    └────────────┘   └────────────┘         │
└─────────────────────────────────────────────────────────────────────────────┘
```

Each section below implements one numbered block.

## Setup

In [ ]:
import json, os, time, warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

API_KEY = os.environ.get("OPENAI_API_KEY", "")

with open("config.json") as f:
    CONFIG = json.load(f)

print(f"LLM model      : {CONFIG['llm_model']}")
print(f"Embedding model : {CONFIG['embedding_model']}")
print(f"Chunk size      : {CONFIG['chunk_size']}")
print(f"Chunk overlap   : {CONFIG['chunk_overlap']}")
print(f"Retriever top-k : {CONFIG['retriever_top_k']}")
print(f"API key set     : {bool(API_KEY)}")

if not API_KEY:
    print("\nNo OPENAI_API_KEY set — simulated responses will be used.")

## Block 1: Load documents

Load the knowledge base — IT policy documents. In production this would come from files, a database, or an API. Here we define them inline.

In [ ]:
from langchain_core.documents import Document

raw_documents = [
    {
        "id": "POL-001",
        "title": "Remote Work Policy",
        "content": (
            "Remote work is available to all full-time employees after completing 90 days of employment. "
            "Employees must use the corporate VPN (GlobalProtect) for all work activities. A dedicated workspace "
            "with reliable internet (minimum 25 Mbps) is required. Remote workers must be available during core "
            "hours (10 AM - 3 PM local time). Equipment provided: laptop, monitor, keyboard, mouse. Home office "
            "stipend of $500 is available annually for ergonomic furniture. Manager approval is required for "
            "international remote work. Remote workers must attend quarterly in-person team meetings."
        ),
    },
    {
        "id": "POL-002",
        "title": "Information Security Policy",
        "content": (
            "All employees must complete annual cybersecurity training by December 31. Passwords must be minimum "
            "12 characters with uppercase, lowercase, number, and special character. Password rotation every 90 "
            "days. Multi-factor authentication is mandatory for all cloud services. Report security incidents to "
            "security@company.com within 1 hour. Data classification: Public, Internal, Confidential, Restricted. "
            "Customer data is always Confidential or Restricted. Laptop encryption (BitLocker/FileVault) is "
            "mandatory. USB storage devices are blocked on corporate laptops. Phishing attempts must be reported "
            "immediately using the Report Phishing button in Outlook."
        ),
    },
    {
        "id": "POL-003",
        "title": "Software Request Process",
        "content": (
            "All software installations require IT department approval. Submit requests through the ServiceNow "
            "catalog. Standard software (Office 365, Slack, Zoom) is pre-approved and auto-installed. Non-standard "
            "software requires manager approval and security review (1-2 business days). Open-source software must "
            "pass license compatibility check. Unapproved software installations will be flagged and removed. "
            "Software license costs above $500 require VP-level budget approval. Development tools (VS Code, Git, "
            "Docker) are pre-approved for engineering teams."
        ),
    },
    {
        "id": "POL-004",
        "title": "Incident Response Procedure",
        "content": (
            "Security incidents must be reported within 1 hour to security@company.com. Severity levels: Critical "
            "(data breach, ransomware), High (unauthorized access, phishing success), Medium (suspicious activity, "
            "policy violation), Low (failed login attempts, spam). Critical and High incidents trigger the Incident "
            "Response Team (IRT). IRT members: CISO, IT Security Lead, Legal, Communications. Post-incident review "
            "within 48 hours. Root cause analysis within 5 business days. All incidents logged in the SIEM system. "
            "Communication to affected parties within 24 hours for Critical incidents."
        ),
    },
    {
        "id": "POL-005",
        "title": "Data Retention Policy",
        "content": (
            "Email retention: 7 years for business communications, 1 year for general. Financial records: 7 years "
            "(regulatory requirement). Customer data: retained while account is active plus 2 years after closure. "
            "Employee records: 7 years after termination. Project files: 3 years after project completion. Backup "
            "retention: daily for 30 days, weekly for 90 days, monthly for 1 year. Data deletion requests processed "
            "within 30 days per GDPR requirements. All retention periods are minimum; departments may retain longer "
            "with documented justification."
        ),
    },
    {
        "id": "POL-006",
        "title": "Acceptable Use Policy",
        "content": (
            "Corporate devices and network are for business use. Limited personal use is acceptable during breaks. "
            "Prohibited: accessing illegal content, cryptocurrency mining, unauthorized file sharing, bypassing "
            "security controls, connecting unauthorized devices. Social media use must comply with the social media "
            "policy. Personal cloud storage (Dropbox, Google Drive personal) must not store corporate data. "
            "Violations may result in disciplinary action up to termination. All network traffic is monitored and "
            "logged for security purposes."
        ),
    },
    {
        "id": "POL-007",
        "title": "Travel and Expense Policy",
        "content": (
            "Business travel requires manager pre-approval. Economy class for flights under 6 hours, business class "
            "for longer flights. Hotel maximum: $200/night domestic, $300/night international. Meal per diem: $75 "
            "domestic, $100 international. Receipts required for expenses over $25. Expense reports due within 10 "
            "business days of trip completion. Corporate credit card must be used when available. Mileage "
            "reimbursement: $0.67 per mile. Conference registration over $1,000 requires director approval."
        ),
    },
    {
        "id": "POL-008",
        "title": "Employee Onboarding",
        "content": (
            "New employee orientation on first day: HR paperwork, IT setup, office tour, team introduction. IT "
            "provisions: laptop, email, Slack, VPN access within 24 hours. Required training in first 30 days: "
            "cybersecurity awareness, code of conduct, anti-harassment, data privacy. Badge access provisioned on "
            "day one. Buddy system: each new hire paired with a tenured team member for 90 days. Probation period: "
            "90 days with mid-point review at 45 days and final review at 90 days. Manager check-in meetings "
            "weekly during probation."
        ),
    },
    {
        "id": "POL-009",
        "title": "Leave and PTO Policy",
        "content": (
            "Full-time employees receive 20 days PTO annually, accrued monthly. Sick leave: 10 days per year, "
            "no carryover. Parental leave: 12 weeks paid for primary caregiver, 4 weeks for secondary. "
            "Bereavement leave: 5 days for immediate family, 3 days for extended family. Jury duty: paid time off "
            "for duration of service. PTO requests require 2 weeks advance notice for 3+ consecutive days. "
            "Unused PTO carries over up to 5 days maximum. PTO blackout periods may apply during critical "
            "business periods with 30 days notice."
        ),
    },
    {
        "id": "POL-010",
        "title": "Code of Conduct",
        "content": (
            "All employees must act with integrity, respect, and professionalism. Conflicts of interest must be "
            "disclosed to HR and direct manager. Gifts from vendors or clients above $50 must be reported. "
            "Harassment, discrimination, and retaliation are strictly prohibited and will result in immediate "
            "investigation. Whistleblower protections apply to all good-faith reports. Confidential reporting "
            "available via the Ethics Hotline (1-800-ETHICS-1). Annual code of conduct acknowledgment required."
        ),
    },
]

# Convert to LangChain Document objects
lc_documents = [
    Document(
        page_content=doc["content"],
        metadata={"id": doc["id"], "title": doc["title"]},
    )
    for doc in raw_documents
]

print(f"[Block 1] Loaded {len(lc_documents)} documents")
for doc in lc_documents:
    print(f"  [{doc.metadata['id']}] {doc.metadata['title']} ({len(doc.page_content)} chars)")

## Block 2: Chunk documents

Split documents into smaller chunks for effective embedding and retrieval. Overlap ensures context is not lost at chunk boundaries.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
    separators=["\n\n", "\n", ". ", " ", ""],
)

all_chunks = splitter.split_documents(lc_documents)

print(f"[Block 2] {len(lc_documents)} documents → {len(all_chunks)} chunks")
print(f"Chunk sizes: min={min(len(c.page_content) for c in all_chunks)}, "
      f"max={max(len(c.page_content) for c in all_chunks)}, "
      f"avg={sum(len(c.page_content) for c in all_chunks)//len(all_chunks)}")

# Show chunk distribution per document
from collections import Counter
chunk_dist = Counter(c.metadata["id"] for c in all_chunks)
print("\nChunks per document:")
for doc_id, count in sorted(chunk_dist.items()):
    title = next(d.metadata["title"] for d in lc_documents if d.metadata["id"] == doc_id)
    print(f"  [{doc_id}] {title}: {count} chunks")

## Block 3: Embed and index in vector store

Embed all chunks using `sentence-transformers` (runs locally, no API) and store in a Chroma vector database.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name=CONFIG["embedding_model"])

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="lab_qa",
)

print(f"[Block 3] Indexed {vectorstore._collection.count()} chunks in Chroma")
print(f"Embedding model: {CONFIG['embedding_model']} (local)")

## Block 4: Retriever

Create a retriever from the vector store and verify it works.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": CONFIG["retriever_top_k"]})

# Verify retrieval
test_query = "What are the password requirements?"
test_docs = retriever.invoke(test_query)

print(f"[Block 4] Retriever created (k={CONFIG['retriever_top_k']})")
print(f"\nTest query: '{test_query}'")
print(f"Retrieved {len(test_docs)} chunks:")
for i, doc in enumerate(test_docs, 1):
    print(f"  {i}. [{doc.metadata['id']}] {doc.page_content[:100]}...")

## Block 5: Prompt template + LLM chain

Build the RAG chain that takes retrieved context and a question, then generates a grounded answer.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(
    model=CONFIG["llm_model"],
    temperature=CONFIG["temperature"],
    api_key=API_KEY or "sk-placeholder",
)
parser = StrOutputParser()

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an internal IT policy assistant. Answer the question using ONLY the provided context. "
        "If the answer is not in the context, respond: 'This information is not available in the current documents.' "
        "Cite the source document ID (e.g., POL-001) when possible. Be concise."
    )),
    ("user", "Context:\n{context}\n\nQuestion: {question}"),
])

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata.get('id', '?')}] {d.page_content}" for d in docs)

# Full RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | parser
)

print("[Block 5] RAG chain assembled: retriever | format_docs | prompt | llm | parser")

## Block 6: Run pipeline — display answer and sources

In [ ]:
from langchain_core.runnables import RunnableLambda

def full_pipeline(question):
    start = time.perf_counter()

    # Retrieve
    docs = retriever.invoke(question)
    context = format_docs(docs)
    sources = [{"id": d.metadata.get("id"), "title": d.metadata.get("title")} for d in docs]

    # Generate
    if API_KEY:
        answer = (rag_prompt | llm | parser).invoke({
            "context": context,
            "question": question,
        })
    else:
        answer = "[Simulated] Answer grounded in retrieved context."

    elapsed = time.perf_counter() - start

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "latency": round(elapsed, 3),
    }

def display_result(result):
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"Sources: {', '.join(s['id'] + ' (' + s['title'] + ')' for s in result['sources'])}")
    print(f"Latency: {result['latency']}s")
    print()

In [ ]:
# Run the full pipeline
print("[Block 6] Running full pipeline...\n")

result = full_pipeline("What are the password requirements?")
display_result(result)

## Test queries

In [ ]:
test_questions = [
    "How do I request new software?",
    "What is the hotel limit for international travel?",
    "How many days of PTO do full-time employees get?",
    "What training is required in the first 30 days?",
    "What should I do if I receive a phishing email?",
    "What is the mileage reimbursement rate?",
    "How long are customer records retained after account closure?",
    "What is the gift value limit from vendors?",
]

simulated_answers = [
    "Submit software requests through the ServiceNow catalog. Standard software is pre-approved; non-standard requires manager approval and security review (1-2 business days). (POL-003)",
    "Hotel maximum for international travel is $300/night. (POL-007)",
    "Full-time employees receive 20 days PTO annually, accrued monthly. (POL-009)",
    "Required training in first 30 days: cybersecurity awareness, code of conduct, anti-harassment, data privacy. (POL-008)",
    "Report phishing attempts immediately using the Report Phishing button in Outlook. (POL-002)",
    "Mileage reimbursement rate is $0.67 per mile. (POL-007)",
    "Customer data is retained while the account is active plus 2 years after closure. (POL-005)",
    "Gifts from vendors or clients above $50 must be reported. (POL-010)",
]

print(f"Testing {len(test_questions)} queries...\n")
for i, q in enumerate(test_questions):
    if API_KEY:
        result = full_pipeline(q)
    else:
        result = {
            "question": q,
            "answer": simulated_answers[i],
            "sources": [{"id": "POL-???", "title": "Simulated"}],
            "latency": 0.5,
        }
    display_result(result)

## Out-of-scope question handling

In [ ]:
out_of_scope = [
    "What is the company's stock price?",
    "Who won the Super Bowl last year?",
    "What is the meaning of life?",
]

print("Out-of-scope queries (should decline to answer):\n")
for q in out_of_scope:
    if API_KEY:
        result = full_pipeline(q)
    else:
        result = {
            "question": q,
            "answer": "This information is not available in the current documents.",
            "sources": [{"id": "N/A", "title": "N/A"}],
            "latency": 0.3,
        }
    display_result(result)

## Evaluation

### Retrieval accuracy (Recall@k)

For each question, check whether the correct source document appears in the retrieved chunks.

In [ ]:
ground_truth = [
    ("What are the password requirements?", "POL-002"),
    ("Can I work remotely from another country?", "POL-001"),
    ("How do I request Photoshop?", "POL-003"),
    ("What is a Critical severity incident?", "POL-004"),
    ("How long are emails retained?", "POL-005"),
    ("Can I mine cryptocurrency on my work laptop?", "POL-006"),
    ("What is the hotel limit for domestic travel?", "POL-007"),
    ("What happens during the 45-day review?", "POL-008"),
    ("How much sick leave do I get per year?", "POL-009"),
    ("What is the Ethics Hotline number?", "POL-010"),
]

hits = 0
print(f"Evaluating retrieval (Recall@{CONFIG['retriever_top_k']}):\n")
for question, expected_id in ground_truth:
    docs = retriever.invoke(question)
    retrieved_ids = [d.metadata.get("id") for d in docs]
    found = expected_id in retrieved_ids
    hits += found
    status = " HIT" if found else "MISS"
    print(f"  [{status}] Q: {question[:50]:50s} Expected: {expected_id} | Got: {retrieved_ids}")

recall = hits / len(ground_truth)
print(f"\nRecall@{CONFIG['retriever_top_k']}: {recall:.0%} ({hits}/{len(ground_truth)})")

### Detailed miss analysis

In [ ]:
# For any misses, show what was retrieved vs expected
misses = []
for question, expected_id in ground_truth:
    docs = retriever.invoke(question)
    retrieved_ids = [d.metadata.get("id") for d in docs]
    if expected_id not in retrieved_ids:
        misses.append({
            "question": question,
            "expected": expected_id,
            "retrieved": retrieved_ids,
            "retrieved_snippets": [d.page_content[:80] for d in docs],
        })

if misses:
    print(f"{len(misses)} miss(es) found:\n")
    for m in misses:
        print(f"  Question: {m['question']}")
        print(f"  Expected: {m['expected']}")
        print(f"  Retrieved: {m['retrieved']}")
        for j, snippet in enumerate(m["retrieved_snippets"]):
            print(f"    {j+1}. {snippet}...")
        print()
else:
    print("No misses — all expected documents found in top-k results.")

### Latency benchmark

In [ ]:
# Measure retrieval-only latency vs full pipeline latency
retrieval_times = []
full_times = []

benchmark_questions = [q for q, _ in ground_truth[:5]]

for q in benchmark_questions:
    # Retrieval only
    start = time.perf_counter()
    _ = retriever.invoke(q)
    retrieval_times.append(time.perf_counter() - start)

    # Full pipeline (if API key available)
    if API_KEY:
        start = time.perf_counter()
        _ = full_pipeline(q)
        full_times.append(time.perf_counter() - start)

avg_retrieval = sum(retrieval_times) / len(retrieval_times)
print(f"Average retrieval latency: {avg_retrieval*1000:.0f} ms")

if full_times:
    avg_full = sum(full_times) / len(full_times)
    print(f"Average full pipeline latency: {avg_full*1000:.0f} ms")
    print(f"LLM overhead: {(avg_full - avg_retrieval)*1000:.0f} ms")
else:
    print("Full pipeline latency: ~500-2000 ms (depends on API)")

## Extending the assistant

### Adding new documents

The pipeline supports incremental updates — add new documents without rebuilding the entire index.

In [ ]:
new_doc = Document(
    page_content=(
        "The company wellness program includes gym membership reimbursement up to $50/month, "
        "annual health screenings covered at 100%, mental health counseling (6 sessions/year), "
        "and ergonomic workspace assessments. Enrollment opens January 1 through the HR portal."
    ),
    metadata={"id": "POL-011", "title": "Wellness Program"},
)

# Chunk and add to existing vectorstore
new_chunks = splitter.split_documents([new_doc])
vectorstore.add_documents(new_chunks)

print(f"Added {len(new_chunks)} chunk(s) for {new_doc.metadata['title']}")
print(f"Total indexed: {vectorstore._collection.count()} chunks")

# Test retrieval for the new topic
if API_KEY:
    result = full_pipeline("Does the company offer gym membership benefits?")
else:
    result = {
        "question": "Does the company offer gym membership benefits?",
        "answer": "Yes, the wellness program includes gym membership reimbursement up to $50/month. (POL-011)",
        "sources": [{"id": "POL-011", "title": "Wellness Program"}],
        "latency": 0.5,
    }
display_result(result)

### Multi-turn conversation

In [ ]:
# Simulate a conversation where follow-up questions build on previous context
conversation = [
    "What is the parental leave policy?",
    "Is bereavement leave also available?",
    "How much advance notice do I need for PTO?",
]

simulated_conv = [
    "Primary caregivers receive 12 weeks paid parental leave, secondary caregivers receive 4 weeks. (POL-009)",
    "Yes, bereavement leave is 5 days for immediate family and 3 days for extended family. (POL-009)",
    "PTO requests require 2 weeks advance notice for 3+ consecutive days. (POL-009)",
]

print("Multi-turn conversation:\n")
for i, q in enumerate(conversation):
    if API_KEY:
        result = full_pipeline(q)
    else:
        result = {
            "question": q,
            "answer": simulated_conv[i],
            "sources": [{"id": "POL-009", "title": "Leave and PTO Policy"}],
            "latency": 0.5,
        }
    print(f"User: {result['question']}")
    print(f"Assistant: {result['answer']}")
    print(f"  [Sources: {', '.join(s['id'] for s in result['sources'])}]")
    print()

## Production wrapper: DocumentQAAssistant

Encapsulate the full pipeline in a reusable class.

In [ ]:
class DocumentQAAssistant:
    def __init__(self, documents, config, api_key):
        self.config = config
        self.api_key = api_key

        # Chunk
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=config["chunk_size"],
            chunk_overlap=config["chunk_overlap"],
        )
        chunks = splitter.split_documents(documents)

        # Embed and index
        embeddings = HuggingFaceEmbeddings(model_name=config["embedding_model"])
        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name="assistant_production",
        )
        self.retriever = self.vectorstore.as_retriever(
            search_kwargs={"k": config["retriever_top_k"]}
        )

        # LLM chain
        self.llm = ChatOpenAI(
            model=config["llm_model"],
            temperature=config["temperature"],
            api_key=api_key or "sk-placeholder",
        )
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", (
                "You are an internal policy assistant. Answer using ONLY the provided context. "
                "If the answer is not in the context, say 'Not available in current documents.' "
                "Cite source document IDs. Be concise."
            )),
            ("user", "Context:\n{context}\n\nQuestion: {question}"),
        ])
        self.parser = StrOutputParser()
        self.query_count = 0

    def ask(self, question):
        self.query_count += 1
        start = time.perf_counter()

        docs = self.retriever.invoke(question)
        context = "\n\n".join(f"[{d.metadata.get('id')}] {d.page_content}" for d in docs)
        sources = [d.metadata.get("id") for d in docs]

        if self.api_key:
            answer = (self.prompt | self.llm | self.parser).invoke({
                "context": context,
                "question": question,
            })
        else:
            answer = "[Simulated] Grounded answer from retrieved context."

        return {
            "question": question,
            "answer": answer,
            "sources": sources,
            "latency": round(time.perf_counter() - start, 3),
            "query_number": self.query_count,
        }

    def add_documents(self, new_docs):
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.config["chunk_size"],
            chunk_overlap=self.config["chunk_overlap"],
        )
        chunks = splitter.split_documents(new_docs)
        self.vectorstore.add_documents(chunks)
        return len(chunks)

    def stats(self):
        return {
            "total_indexed": self.vectorstore._collection.count(),
            "queries_processed": self.query_count,
            "model": self.config["llm_model"],
            "embedding_model": self.config["embedding_model"],
        }

print("DocumentQAAssistant class defined.")

In [ ]:
# Instantiate and test
assistant = DocumentQAAssistant(lc_documents, CONFIG, API_KEY)

# Run queries
for q in ["What is the probation period?", "How do I report a security incident?"]:
    result = assistant.ask(q)
    print(f"Q{result['query_number']}: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"Sources: {result['sources']} ({result['latency']}s)\n")

# Stats
print("Assistant stats:", assistant.stats())

## Success criteria

- [x] **Block 1**: Documents loaded and converted to LangChain `Document` objects
- [x] **Block 2**: Documents chunked with `RecursiveCharacterTextSplitter`
- [x] **Block 3**: Chunks embedded and indexed in Chroma vector store
- [x] **Block 4**: Retriever returns relevant chunks for test queries
- [x] **Block 5**: RAG chain generates answers grounded in retrieved context
- [x] **Block 6**: Answers displayed with source document attribution
- [x] **Evaluation**: Recall@k measured against ground truth
- [x] **Extensible**: New documents can be added without full rebuild
- [x] **Production class**: `DocumentQAAssistant` wraps the full pipeline

## Cleanup

In [ ]:
try:
    vectorstore._client.delete_collection("lab_qa")
    assistant.vectorstore._client.delete_collection("assistant_production")
    print("Temporary Chroma collections removed.")
except Exception:
    pass

print("Lab complete.")